# Conquer Mobile — Deep 3D Asset Inspection (One Cell)

تشغيل خلية الكود الوحيدة. تسحب الأصول مباشرة من GitHub Release الثابت، وتفحص النماذج هندسيًا وبنيويًا، وتتحقق من بيانات الخامات والـTextures والـSkeletons والـAnimations والاعتماديات الخارجية، وتنشئ صور معاينة ثابتة وContact Sheet، ثم ترفع خمسة تقارير/حزم إلى نفس الإصدار.

**لا Google Drive، لا GPU، لا طلب مسار أو رابط أو اختيار ملفات.** تحتاج Secret باسم `GITHUB_TOKEN` مع صلاحية تعديل Release assets وNotebook access مفعّل. التشغيل على CPU عادي.

In [ ]:
%pip -q install requests trimesh matplotlib pillow

import os, re, io, gc, json, csv, html, base64, hashlib, shutil, stat, zipfile, requests, struct, xml.etree.ElementTree as ET
from pathlib import Path, PurePosixPath
from collections import Counter
from PIL import Image, ImageOps, ImageDraw
from google.colab import userdata
import trimesh
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Fixed GitHub source. No Drive, path prompt, file picker, or URL input.
OWNER = "ZzwahwahzZ"
REPO = "Conquer-Mobile-Assets."
TAG = "mobile-extracted-assets"
BUNDLE_NAME = "extracted-assets-001.zip"
WORK = Path("/content/conquer_mobile_deep_audit")
ARCHIVE = WORK / BUNDLE_NAME
EXTRACTED = WORK / "extracted"
REPORTS = WORK / "deep_audit"
PREVIEWS = REPORTS / "previews"
API = f"https://api.github.com/repos/{OWNER}/{REPO}"
MODEL_EXTS = {".glb", ".gltf", ".obj", ".fbx", ".dae", ".blend", ".3ds", ".stl", ".ply"}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".tga", ".bmp", ".dds"}
CHUNK = 8 * 1024 * 1024
SAFETY_BYTES = 700 * 1024 * 1024
MAX_PREVIEW_FACES = 9000

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()

def github_headers(token):
    return {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28"}

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    count = total = 0
    with zipfile.ZipFile(zip_path) as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"ZIP CRC failed: {bad}")
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            member = PurePosixPath(name)
            if member.is_absolute() or ".." in member.parts or not member.parts:
                raise RuntimeError(f"Unsafe ZIP path: {info.filename}")
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_ISLNK(mode):
                raise RuntimeError(f"Symlink rejected: {info.filename}")
            target = (destination / Path(*member.parts)).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"ZIP path escapes destination: {info.filename}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=CHUNK)
            count += 1
            total += info.file_size
    return count, total

def upload_report(session, release, path):
    upload_url = release["upload_url"].split("{", 1)[0]
    assets_url = f'{API}/releases/{release["id"]}/assets'
    assets = session.get(assets_url, timeout=60)
    assets.raise_for_status()
    old = next((a for a in assets.json() if a["name"] == path.name), None)
    if old:
        d = session.delete(old["url"], timeout=60)
        d.raise_for_status()
    with open(path, "rb") as f:
        r = session.post(upload_url, params={"name": path.name},
                         headers={**session.headers, "Content-Type": "application/octet-stream"},
                         data=f, timeout=(60, 1200))
    r.raise_for_status()
    a = r.json()
    if a.get("name") != path.name or a.get("size") != path.stat().st_size:
        raise RuntimeError(f"Upload verification failed: {path.name}")
    print(f"UPLOADED + VERIFIED: {a['name']} ({a['size']:,} bytes)")

def parse_glb(path):
    result = {"nodes": 0, "skins": 0, "animations": 0, "materials": 0,
              "textures": 0, "images": 0, "meshes": 0, "embedded_images": 0,
              "external_images": [], "gltf_error": ""}
    try:
        data = path.read_bytes()
        if len(data) < 20 or data[:4] != b"glTF":
            raise ValueError("Invalid GLB header")
        version, total_len = struct.unpack_from("<II", data, 4)
        if version != 2 or total_len > len(data):
            raise ValueError(f"Unexpected GLB version/length: {version}/{total_len}")
        offset, doc = 12, None
        while offset + 8 <= min(total_len, len(data)):
            length, kind = struct.unpack_from("<I4s", data, offset)
            offset += 8
            chunk = data[offset:offset+length]
            offset += length
            if kind == b"JSON":
                doc = json.loads(chunk.rstrip(b"\x00 \t\r\n").decode("utf-8"))
                break
        if doc is None:
            raise ValueError("GLB JSON chunk not found")
        for key, field in [("nodes","nodes"),("skins","skins"),("animations","animations"),
                           ("materials","materials"),("textures","textures"),("images","images"),("meshes","meshes")]:
            result[key] = len(doc.get(field, []))
        for im in doc.get("images", []):
            uri = im.get("uri")
            if uri:
                result["external_images"].append(uri)
            elif "bufferView" in im:
                result["embedded_images"] += 1
        result["gltf_error"] = ""
    except Exception as e:
        result["gltf_error"] = f"{type(e).__name__}: {str(e)[:300]}"
    return result

def parse_gltf(path):
    result = {"nodes": 0, "skins": 0, "animations": 0, "materials": 0,
              "textures": 0, "images": 0, "meshes": 0, "embedded_images": 0,
              "external_images": [], "gltf_error": ""}
    try:
        doc = json.loads(path.read_text(encoding="utf-8"))
        for key, field in [("nodes","nodes"),("skins","skins"),("animations","animations"),
                           ("materials","materials"),("textures","textures"),("images","images"),("meshes","meshes")]:
            result[key] = len(doc.get(field, []))
        for im in doc.get("images", []):
            if im.get("uri"): result["external_images"].append(im["uri"])
            elif "bufferView" in im: result["embedded_images"] += 1
    except Exception as e:
        result["gltf_error"] = f"{type(e).__name__}: {str(e)[:300]}"
    return result

def parse_obj(path):
    info = {"mtllibs": [], "texture_refs": [], "missing_dependencies": []}
    try:
        for line in path.read_text(errors="replace").splitlines():
            s = line.strip()
            if s.lower().startswith("mtllib "):
                info["mtllibs"].extend(s.split(None, 1)[1].split())
        for lib in info["mtllibs"]:
            mtl = (path.parent / lib).resolve()
            if not mtl.exists():
                info["missing_dependencies"].append(lib)
                continue
            for line in mtl.read_text(errors="replace").splitlines():
                s = line.strip()
                if re.match(r"^(map_\w+|bump|disp|decal|norm)\s+", s, re.I):
                    ref = s.split()[-1]
                    info["texture_refs"].append(ref)
                    if not (mtl.parent / ref).exists():
                        info["missing_dependencies"].append(f"{lib} -> {ref}")
    except Exception as e:
        info["parse_error"] = f"{type(e).__name__}: {str(e)[:250]}"
    return info

def parse_dae(path):
    info = {"dae_geometries": 0, "dae_controllers": 0, "dae_animations": 0,
            "dae_images": 0, "dae_image_refs": [], "missing_dependencies": []}
    try:
        root = ET.parse(path).getroot()
        for el in root.iter():
            tag = el.tag.split("}")[-1]
            if tag == "geometry": info["dae_geometries"] += 1
            elif tag == "controller": info["dae_controllers"] += 1
            elif tag == "animation": info["dae_animations"] += 1
            elif tag == "init_from" and el.text:
                ref = el.text.strip()
                info["dae_images"] += 1
                info["dae_image_refs"].append(ref)
                if not (path.parent / ref).exists():
                    info["missing_dependencies"].append(ref)
    except Exception as e:
        info["parse_error"] = f"{type(e).__name__}: {str(e)[:250]}"
    return info

def make_preview(meshes, out_path):
    # Static geometry thumbnail; this does not preview rigged animation or materials.
    usable = [m for m in meshes if len(m.vertices) and len(m.faces)]
    if not usable:
        return False
    verts = []
    faces = []
    offset = 0
    for m in usable:
        v = m.vertices
        if not len(v): continue
        verts.append(v)
        f = m.faces
        if len(f) > MAX_PREVIEW_FACES:
            step = max(1, len(f)//MAX_PREVIEW_FACES)
            f = f[::step][:MAX_PREVIEW_FACES]
        faces.append(f + offset)
        offset += len(v)
    if not verts or not faces: return False
    V = __import__("numpy").vstack(verts)
    F = __import__("numpy").vstack(faces)
    if not __import__("numpy").isfinite(V).all(): return False
    lo, hi = V.min(axis=0), V.max(axis=0)
    span = hi-lo
    span[span == 0] = 1.0
    V = (V - (lo+hi)/2) / max(float(span.max()), 1e-9)
    tri = V[F]
    fig = plt.figure(figsize=(3.2, 2.6), dpi=90)
    ax = fig.add_subplot(111, projection="3d")
    poly = Poly3DCollection(tri, linewidths=0.03, alpha=0.95)
    ax.add_collection3d(poly)
    ax.set_xlim(-0.6,0.6); ax.set_ylim(-0.6,0.6); ax.set_zlim(-0.6,0.6)
    ax.view_init(elev=22, azim=35)
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    fig.savefig(out_path, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    return True

print("Preparing deep-audit workspace...")
WORK.mkdir(parents=True, exist_ok=True)
if ARCHIVE.exists(): ARCHIVE.unlink()
if EXTRACTED.exists(): shutil.rmtree(EXTRACTED)
if REPORTS.exists(): shutil.rmtree(REPORTS)
EXTRACTED.mkdir(parents=True); PREVIEWS.mkdir(parents=True)

print("Resolving the fixed GitHub Release asset...")
rel0 = requests.get(f"{API}/releases/tags/{TAG}", timeout=60)
rel0.raise_for_status()
release_public = rel0.json()
asset = next((a for a in release_public.get("assets", []) if a["name"] == BUNDLE_NAME), None)
if not asset: raise FileNotFoundError(f"{BUNDLE_NAME} missing from release {TAG}")
expected_size = int(asset["size"])
free = shutil.disk_usage("/content").free
print(f"Source: {release_public['html_url']}")
print(f"Archive size: {expected_size:,} | Free disk: {free:,}")
if free < expected_size + SAFETY_BYTES:
    raise RuntimeError("Insufficient disk space; extraction was not started.")

print("Downloading source ZIP from GitHub...")
h = hashlib.sha256(); downloaded = 0
with requests.get(asset["browser_download_url"], stream=True, timeout=(30,300)) as r:
    r.raise_for_status()
    with open(ARCHIVE,"wb") as out:
        for block in r.iter_content(CHUNK):
            if block:
                out.write(block); h.update(block); downloaded += len(block)
if downloaded != expected_size: raise RuntimeError("Downloaded byte count does not match GitHub.")
archive_hash = h.hexdigest()
print(f"DOWNLOAD VERIFIED: {downloaded:,} bytes | SHA-256 {archive_hash}")

with zipfile.ZipFile(ARCHIVE) as z:
    bad = z.testzip()
    if bad: raise RuntimeError(f"ZIP CRC failure: {bad}")
    expanded = sum(x.file_size for x in z.infolist() if not x.is_dir())
if shutil.disk_usage("/content").free < expanded + SAFETY_BYTES:
    raise RuntimeError("Insufficient free disk for extraction.")
print("Safely extracting...")
file_count, expanded_bytes = safe_extract(ARCHIVE, EXTRACTED)
print(f"EXTRACTED: {file_count} files | {expanded_bytes:,} bytes")

all_files = [p for p in EXTRACTED.rglob("*") if p.is_file()]
image_files = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS]
models = sorted([p for p in all_files if p.suffix.lower() in MODEL_EXTS],
                key=lambda p:p.relative_to(EXTRACTED).as_posix().casefold())
print(f"Deep audit targets: {len(models)} models | {len(image_files)} image/texture files")

rows = []
for i, path in enumerate(models, 1):
    rel = path.relative_to(EXTRACTED).as_posix()
    row = {"path":rel,"extension":path.suffix.lower(),"size_bytes":path.stat().st_size,
           "sha256":sha256_file(path),"parser_status":"not_tested","geometry_count":0,
           "vertices":0,"faces":0,"finite_vertices":True,"zero_extent_axes":0,
           "materials_detected":0,"visual_kind":"","nodes":0,"skins":0,"animations":0,
           "gltf_materials":0,"gltf_textures":0,"gltf_images":0,"embedded_images":0,
           "external_image_refs":[],"external_refs_missing":[],"dae_controllers":0,
           "dae_animations":0,"obj_mtllibs":[],"obj_texture_refs":[],"missing_dependencies":[],
           "preview":"", "preview_status":"not_created","godot_hint":"","notes":[]}
    loaded = None
    try:
        loaded = trimesh.load(str(path), force="scene", process=False)
        if isinstance(loaded, trimesh.Trimesh): geoms = [loaded]
        elif isinstance(loaded, trimesh.Scene): geoms = list(loaded.geometry.values())
        else: geoms = []
        meshes = [g for g in geoms if isinstance(g,trimesh.Trimesh)]
        row["geometry_count"] = len(geoms)
        row["vertices"] = sum(len(m.vertices) for m in meshes)
        row["faces"] = sum(len(m.faces) for m in meshes)
        import numpy as np
        row["finite_vertices"] = all(np.isfinite(m.vertices).all() for m in meshes)
        row["zero_extent_axes"] = sum(1 for m in meshes if np.any(np.ptp(m.vertices,axis=0)==0)) if meshes else 0
        mats = []
        kinds = []
        for g in geoms:
            vis = getattr(g,"visual",None)
            if vis is not None:
                kinds.append(type(vis).__name__)
                mat = getattr(vis,"material",None)
                if mat is not None: mats.append(type(mat).__name__)
        row["visual_kind"] = ",".join(sorted(set(kinds)))
        row["materials_detected"] = len(set(mats))
        row["parser_status"] = "parsed_mesh" if meshes and row["vertices"] and row["faces"] else "parsed_no_usable_mesh"
        if meshes and row["finite_vertices"]:
            preview_name = f"{i:03d}_{hashlib.sha1(rel.encode()).hexdigest()[:10]}.jpg"
            preview_path = PREVIEWS / preview_name
            if make_preview(meshes, preview_path):
                row["preview"] = preview_name
                row["preview_status"] = "static_geometry_thumbnail"
        elif not row["finite_vertices"]:
            row["notes"].append("Non-finite vertex coordinates detected")
    except Exception as e:
        row["parser_status"] = "parser_error_or_unsupported"
        row["notes"].append(f"{type(e).__name__}: {str(e)[:350]}")
        meshes = []
    ext = path.suffix.lower()
    if ext == ".glb":
        g = parse_glb(path)
        for k in ["nodes","skins","animations","materials","textures","images","embedded_images"]:
            row[k if k in ("nodes","skins","animations") else "gltf_"+k] = g[k]
        row["external_image_refs"] = g["external_images"]
        row["missing_dependencies"].extend(g["external_images"])
        if g["gltf_error"]: row["notes"].append(g["gltf_error"])
    elif ext == ".gltf":
        g = parse_gltf(path)
        for k in ["nodes","skins","animations","materials","textures","images","embedded_images"]:
            row[k if k in ("nodes","skins","animations") else "gltf_"+k] = g[k]
        row["external_image_refs"] = g["external_images"]
        for ref in g["external_images"]:
            if not (path.parent/ref).exists(): row["external_refs_missing"].append(ref)
        for b in json.loads(path.read_text(encoding="utf-8")).get("buffers",[]):
            u=b.get("uri")
            if u and not (path.parent/u).exists(): row["external_refs_missing"].append(u)
    elif ext == ".obj":
        d = parse_obj(path)
        row["obj_mtllibs"] = d["mtllibs"]; row["obj_texture_refs"] = d["texture_refs"]
        row["missing_dependencies"].extend(d["missing_dependencies"])
    elif ext == ".dae":
        d = parse_dae(path)
        row["dae_controllers"] = d["dae_controllers"]; row["dae_animations"] = d["dae_animations"]
        row["missing_dependencies"].extend(d["missing_dependencies"])
    if ext in (".glb",".gltf"):
        if row["skins"] or row["animations"]:
            row["godot_hint"] = "Animated/rigged glTF candidate — preserve skeleton and animation; verify in Godot"
        else:
            row["godot_hint"] = "Static glTF candidate — preferred direct Godot import test"
    elif ext == ".obj":
        row["godot_hint"] = "Static mesh candidate — check MTL and texture dependencies; no skeletal animation"
    elif ext == ".dae":
        row["godot_hint"] = "Collada candidate — verify controller/animation import; consider conversion only after inspection"
    else:
        row["godot_hint"] = "Requires format-specific importer/conversion validation before Godot use"
    if row["missing_dependencies"] or row["external_refs_missing"]:
        row["notes"].append("One or more referenced external dependencies were not found beside the model")
    if row["faces"] == 0:
        row["notes"].append("No triangle faces were reported by the geometry parser")
    rows.append(row)
    if i % 10 == 0 or i == len(models):
        print(f"Deep-audited {i}/{len(models)}: {path.name} | {row['parser_status']} | skins={row['skins']} animations={row['animations']} textures={row['gltf_textures']}")
    if loaded is not None: del loaded
    gc.collect()

# Build contact sheet from generated thumbnails.
thumb_paths = sorted(PREVIEWS.glob("*.jpg"))
thumb_w, thumb_h, label_h, cols = 240, 205, 38, 4
tiles = []
row_by_preview = {r["preview"]:r for r in rows if r["preview"]}
for tp in thumb_paths:
    im = Image.open(tp).convert("RGB")
    im.thumbnail((thumb_w-12,thumb_h-12))
    tile = Image.new("RGB",(thumb_w,thumb_h+label_h),"white")
    tile.paste(im,((thumb_w-im.width)//2,(thumb_h-im.height)//2))
    dr = ImageDraw.Draw(tile)
    rr = row_by_preview.get(tp.name,{})
    label = Path(rr.get("path","")).name[:34]
    dr.text((5,thumb_h+3),label,fill="black")
    dr.text((5,thumb_h+19),rr.get("extension",""),fill="black")
    tiles.append(tile)
if tiles:
    sheet = Image.new("RGB",(cols*thumb_w,((len(tiles)+cols-1)//cols)*(thumb_h+label_h)),(225,225,225))
    for j,t in enumerate(tiles): sheet.paste(t,((j%cols)*thumb_w,(j//cols)*(thumb_h+label_h)))
else:
    sheet = Image.new("RGB",(640,240),"white")
    ImageDraw.Draw(sheet).text((20,20),"No preview images could be generated",fill="black")
contact_path = REPORTS / "model-contact-sheet.jpg"
sheet.save(contact_path,quality=88,optimize=True)

# Archive all individual previews so the gallery can be inspected separately.
preview_zip = REPORTS / "model-previews.zip"
with zipfile.ZipFile(preview_zip,"w",zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in thumb_paths: z.write(p,arcname=p.name)

status_counts = dict(Counter(r["parser_status"] for r in rows))
rigged = sum(1 for r in rows if r["skins"]>0)
animated = sum(1 for r in rows if r["animations"]>0 or r["dae_animations"]>0)
with_materials = sum(1 for r in rows if r["materials_detected"]>0 or r["gltf_materials"]>0)
with_textures = sum(1 for r in rows if r["gltf_textures"]>0 or r["obj_texture_refs"] or r["gltf_images"]>0)
missing_refs = sum(1 for r in rows if r["missing_dependencies"] or r["external_refs_missing"])
preview_count = sum(1 for r in rows if r["preview"])
summary = {
 "source_release":release_public["html_url"],"source_asset":BUNDLE_NAME,
 "source_size_bytes":downloaded,"source_sha256":archive_hash,
 "extracted_files":file_count,"extracted_bytes":expanded_bytes,
 "model_count":len(rows),"image_texture_file_count":len(image_files),
 "parser_status_counts":status_counts,"rigged_glb_count":rigged,
 "animated_model_count":animated,"models_with_material_metadata":with_materials,
 "models_with_texture_metadata_or_refs":with_textures,
 "models_with_missing_external_refs":missing_refs,"static_preview_count":preview_count,
 "models":rows,
 "limitations":"Static thumbnails display geometry only, not original materials, texture maps, lighting, or animation. A parser success is not proof of visual correctness or Godot runtime compatibility. FBX/BLEND may need specialized importers. This report is an inspection aid, not a legal or provenance assessment."
}
json_path=REPORTS/"deep-3d-asset-audit.json"
csv_path=REPORTS/"deep-3d-asset-audit.csv"
html_path=REPORTS/"deep-3d-asset-audit.html"
json_path.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
fields=list(rows[0].keys()) if rows else ["path","extension","parser_status"]
with open(csv_path,"w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=fields,extrasaction="ignore"); w.writeheader()
    for r in rows:
        rr=dict(r)
        for k,v in rr.items():
            if isinstance(v,(list,dict)): rr[k]=json.dumps(v,ensure_ascii=False)
        w.writerow(rr)

sheet_b64=base64.b64encode(contact_path.read_bytes()).decode("ascii")
tr_rows=[]
for r in rows:
    tr_rows.append("<tr><td>"+html.escape(r["path"])+"</td><td>"+html.escape(r["parser_status"])+"</td><td>"+
      str(r["vertices"])+"</td><td>"+str(r["faces"])+"</td><td>"+str(r["skins"])+"</td><td>"+
      str(r["animations"]+r["dae_animations"])+"</td><td>"+str(r["gltf_textures"])+"</td><td>"+
      html.escape("; ".join(r["missing_dependencies"]+r["external_refs_missing"]))+"</td><td>"+
      html.escape(r["godot_hint"])+"</td></tr>")
html_doc=f"""<!doctype html><html><head><meta charset="utf-8"><title>Deep 3D Asset Audit</title>
<style>body{{font:14px Arial;margin:24px}}table{{border-collapse:collapse;width:100%;font-size:12px}}
th,td{{border:1px solid #ccc;padding:6px;text-align:left;vertical-align:top}}th{{position:sticky;top:0;background:#eee}}
img{{max-width:100%;height:auto}}tr:nth-child(even){{background:#f7f7f7}}</style></head><body>
<h1>Conquer Mobile — Deep 3D Asset Audit</h1>
<p>Models: {len(rows)} | Preview images: {preview_count} | Rigged GLB: {rigged} | Animated: {animated}</p>
<pre>{html.escape(json.dumps({k:v for k,v in summary.items() if k not in ("models","limitations")},ensure_ascii=False,indent=2))}</pre>
<h2>Static geometry contact sheet</h2><img src="data:image/jpeg;base64,{sheet_b64}">
<h2>Per-model technical inspection</h2><p>{html.escape(summary["limitations"])}</p>
<table><thead><tr><th>Model path</th><th>Parser</th><th>Vertices</th><th>Faces</th><th>Skins</th><th>Animations</th><th>Textures</th><th>Missing refs</th><th>Godot hint</th></tr></thead>
<tbody>{''.join(tr_rows)}</tbody></table></body></html>"""
html_path.write_text(html_doc,encoding="utf-8")

print("\nDEEP AUDIT SUMMARY")
print(json.dumps({k:v for k,v in summary.items() if k not in ("models","limitations")},ensure_ascii=False,indent=2))
print("Created:",json_path.name,csv_path.name,html_path.name,contact_path.name,preview_zip.name)

print("Connecting to GitHub and publishing all deep-audit outputs...")
token=userdata.get("GITHUB_TOKEN")
if not token: raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets and enable Notebook access.")
session=requests.Session(); session.headers.update(github_headers(token))
who=session.get("https://api.github.com/user",timeout=60); who.raise_for_status()
print("Authenticated to GitHub as:",who.json().get("login"))
repo=session.get(API,timeout=60); repo.raise_for_status()
rel=session.get(f"{API}/releases/tags/{TAG}",timeout=60); rel.raise_for_status()
auth_release=rel.json()
outputs=[json_path,csv_path,html_path,contact_path,preview_zip]
for p in outputs: upload_report(session,auth_release,p)
verify=session.get(f'{API}/releases/{auth_release["id"]}/assets',timeout=60); verify.raise_for_status()
remote={a["name"]:a for a in verify.json()}
for p in outputs:
    a=remote.get(p.name)
    if not a or a["size"]!=p.stat().st_size:
        raise RuntimeError(f"Final GitHub verification failed: {p.name}")
print("\nSUCCESS: deep technical inspection completed; all five outputs uploaded and verified.")
print(f"Release: https://github.com/{OWNER}/{REPO}/releases/tag/{TAG}")
print(f"Models inspected: {len(rows)} | static previews: {preview_count} | rigged: {rigged} | animated: {animated}")
